# Simple Agent (Notebook Version)

Notebook edition of `agent.py` — a minimal Claude-powered agent with a manual tool-use loop, running on `claude-haiku-4-5`.

Run the cells in order. The last cell gives you a chat loop; re-running the "send a message" cell keeps the conversation going since `agent.messages` persists in the notebook's memory.

## 1. Install dependencies

In [ ]:
%pip install -q anthropic

## 2. Set your API key

Uses `getpass` so the key is typed into a hidden prompt instead of being saved in plain text in the notebook file.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")

## 3. Define tools, the agent loop, and the `Agent` class

Includes a calculator, current time, Anthropic's built-in `web_search` server-tool (runs on Anthropic's infra — no extra API key needed), and a file-backed `remember`/`recall` memory store.

Also enforces a spending cap: `max_tokens` limits one reply's length (default 1024, override via `AGENT_MAX_TOKENS`), and `MAX_COST_USD` (default \$0.20, override via `AGENT_MAX_COST_USD`) stops the agent once this kernel session has spent that much.

In [ ]:
import datetime
import json
import os

import anthropic

MODEL = "claude-haiku-4-5"
MAX_TOKENS = int(os.environ.get("AGENT_MAX_TOKENS", "1024"))

# claude-haiku-4-5 pricing, $/1M tokens - update if you switch models.
INPUT_COST_PER_MTOK = 1.00
OUTPUT_COST_PER_MTOK = 5.00

# Hard spending cap for this notebook kernel session.
MAX_COST_USD = float(os.environ.get("AGENT_MAX_COST_USD", "0.20"))


class BudgetExceededError(RuntimeError):
    pass


SYSTEM_PROMPT = (
    "You are a helpful personal assistant with access to tools: a calculator, "
    "the current time, web search, and long-term memory. When the user shares "
    "a fact or preference worth keeping for future conversations, call "
    "'remember'. Use 'web_search' for current events or anything you're not "
    "sure about. Otherwise reply directly."
)

DEFAULT_MEMORY_PATH = ".agent_memory.json"

TOOLS = [
    {
        "name": "get_current_time",
        "description": "Get the current date and time.",
        "input_schema": {
            "type": "object",
            "properties": {},
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression, e.g. '2 + 2 * 3'.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "An arithmetic expression using +, -, *, /, and parentheses.",
                },
            },
            "required": ["expression"],
        },
    },
    {
        "name": "remember",
        "description": "Save a fact or preference about the user to long-term memory.",
        "input_schema": {
            "type": "object",
            "properties": {
                "fact": {
                    "type": "string",
                    "description": "A concise fact to remember, e.g. 'Prefers metric units.'",
                },
            },
            "required": ["fact"],
        },
    },
    {
        "name": "recall",
        "description": "List everything currently stored in long-term memory.",
        "input_schema": {
            "type": "object",
            "properties": {},
        },
    },
    {
        "type": "web_search_20260209",
        "name": "web_search",
        "max_uses": 3,
        "allowed_callers": ["direct"],
    },
]


def get_current_time() -> str:
    return datetime.datetime.now().isoformat()


def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: expression contains disallowed characters."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Error: {exc}"


def load_memory(memory_path: str) -> list:
    if os.path.exists(memory_path):
        with open(memory_path) as f:
            return json.load(f)
    return []


def save_memory(memory_path: str, memory: list) -> None:
    with open(memory_path, "w") as f:
        json.dump(memory, f, indent=2)


def remember(fact: str, memory_path: str) -> str:
    memory = load_memory(memory_path)
    memory.append(fact)
    save_memory(memory_path, memory)
    return f"Remembered: {fact}"


def recall(memory_path: str) -> str:
    memory = load_memory(memory_path)
    if not memory:
        return "No memories stored yet."
    return "\n".join(f"- {fact}" for fact in memory)


def execute_tool(name: str, tool_input: dict, memory_path: str) -> str:
    if name == "get_current_time":
        return get_current_time()
    if name == "calculator":
        return calculator(tool_input["expression"])
    if name == "remember":
        return remember(tool_input["fact"], memory_path)
    if name == "recall":
        return recall(memory_path)
    return f"Error: unknown tool '{name}'"


MAX_PAUSE_RESUMES = 10


class Agent:
    """A minimal conversational agent that can call tools in a loop."""

    def __init__(
        self,
        client: anthropic.Anthropic | None = None,
        memory_path: str | None = None,
    ):
        self.client = client or anthropic.Anthropic()
        self.messages: list[dict] = []
        self.memory_path = memory_path or os.environ.get(
            "AGENT_MEMORY_PATH", DEFAULT_MEMORY_PATH
        )
        self.total_cost_usd = 0.0

    def _system_prompt(self) -> str:
        facts = load_memory(self.memory_path)
        if not facts:
            return SYSTEM_PROMPT
        facts_block = "\n".join(f"- {fact}" for fact in facts)
        return f"{SYSTEM_PROMPT}\n\nThings you remember about the user:\n{facts_block}"

    def send(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})

        resumes = 0
        while True:
            if self.total_cost_usd >= MAX_COST_USD:
                raise BudgetExceededError(
                    f"Session cost ${self.total_cost_usd:.4f} has reached the "
                    f"${MAX_COST_USD:.4f} cap (AGENT_MAX_COST_USD). Raise the "
                    "cap or start a new session to continue."
                )

            response = self.client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=self._system_prompt(),
                tools=TOOLS,
                messages=self.messages,
            )
            self.total_cost_usd += (
                response.usage.input_tokens * INPUT_COST_PER_MTOK
                + response.usage.output_tokens * OUTPUT_COST_PER_MTOK
            ) / 1_000_000
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "pause_turn":
                # A server-side tool (e.g. web_search) hit its per-turn
                # iteration limit; resend as-is to let Claude continue.
                resumes += 1
                if resumes > MAX_PAUSE_RESUMES:
                    break
                continue

            if response.stop_reason != "tool_use":
                break

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input, self.memory_path)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )
            self.messages.append({"role": "user", "content": tool_results})

        return "".join(
            block.text for block in response.content if block.type == "text"
        )


## 4. Create the agent

In [ ]:
agent = Agent()
print(f"Agent ready (cap ${MAX_COST_USD:.4f} for this kernel session)")

## 5. Send a message

Edit the string below and re-run this cell to keep chatting — `agent.messages` keeps the history between runs, so context carries over.

In [ ]:
reply = agent.send("Hi! What's 12 * 7, and what time is it right now?")
print(reply)
print(f"(session cost so far: ~${agent.total_cost_usd:.4f})")

## 5b. Try memory and web search

These persist to `.agent_memory.json` next to the notebook, so run the first cell once, restart the kernel, and the second cell will still recall it.

In [ ]:
print(agent.send("Remember that I prefer dark mode in every app."))
print(f"(session cost so far: ~${agent.total_cost_usd:.4f})")

In [ ]:
print(agent.send("What do you remember about my preferences, and what is one recent AI news headline?"))
print(f"(session cost so far: ~${agent.total_cost_usd:.4f})")

## 6. Optional: interactive chat loop

Run this cell to chat back and forth in the notebook's input prompt. Type `exit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break
    try:
        reply = agent.send(user_input)
    except BudgetExceededError as exc:
        print(f"Agent: [stopped] {exc}")
        break
    print(f"Agent: {reply}  (session cost so far: ~${agent.total_cost_usd:.4f})")